# Part C — Joining, Reliability Feature Engineering & PostgreSQL (Supabase) Storage

**Run this AFTER `02_PySpark_Load_Clean_Partition.ipynb`** -- it reads the flattened CSVs directly and produces the final merged, route-level analysis table + PostgreSQL database (hosted on Supabase).

**Before running:** make sure you've run `pip install psycopg2-binary` in your terminal first.


### 11. Aggregate each dataset to route level (using Pandas — small aggregation, justified per brief's tool-choice guidance)

In [ ]:
import pandas as pd

# Re-load the flattened CSVs directly (Pandas is appropriate here since we're aggregating
# down to ~25 route-level rows -- a small final aggregation step, not large-scale transformation)
tt_pd = pd.read_csv(r'D:\Dataset\timetables_flat.csv', low_memory=False)
fr_pd = pd.read_csv(r'D:\Dataset\fares_flat.csv')
di_pd = pd.read_csv(r'D:\Dataset\catalogue\disruptions_data_catalogue.csv')
di_pd = di_pd[di_pd['Organisation'] == 'West of England']

# Fares: average price per route
fares_agg = fr_pd.groupby('route').agg(avg_fare=('price_gbp', 'mean')).reset_index()
fares_agg['route'] = fares_agg['route'].astype(str)

# Timetables: trip count per route (proxy for service volume)
tt_agg = (tt_pd.groupby('line_name')
          .agg(trip_count=('vehicle_journey_code', 'nunique'))
          .reset_index()
          .rename(columns={'line_name': 'route'}))
tt_agg['route'] = tt_agg['route'].astype(str)

# Disruptions: count per route -- note 'Services affected' is stored as float (e.g. 1.0),
# so it must be converted via float -> int -> str to match Timetables' route format
di_agg = di_pd.groupby('Services affected').size().reset_index(name='disruption_count')
di_agg = di_agg.rename(columns={'Services affected': 'route'})
di_agg['route'] = di_agg['route'].astype(float).astype(int).astype(str)

print('Fares routes:', len(fares_agg), '| Timetables routes:', len(tt_agg), '| Disruption routes:', len(di_agg))

### 12. Merge into one analysis table and engineer the reliability metric

`reliability_score = disruption_count / trip_count` -- lower means more reliable (fewer disruptions relative to how often the route runs).

In [ ]:
merged = tt_agg.merge(fares_agg, on='route', how='inner').merge(di_agg, on='route', how='left')
merged['disruption_count'] = merged['disruption_count'].fillna(0)
merged['reliability_score'] = merged['disruption_count'] / merged['trip_count']

print('Merged routes:', len(merged))
print('Routes with disruption data:', (merged['disruption_count'] > 0).sum())
merged.sort_values('disruption_count', ascending=False)

In [ ]:
print('Correlation matrix (Fare vs Reliability):')
merged[['avg_fare', 'disruption_count', 'reliability_score']].corr()

### 13. Store in Supabase (PostgreSQL) -- relational database requirement

Uses parameterised queries throughout (`%s` placeholders) -- no string concatenation -- to satisfy the brief's security requirement. Connects to a cloud-hosted PostgreSQL database via Supabase, avoiding local database installation.

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="db.fdvxpwnyptbqsktalwqd.supabase.co",
    port=5432,
    database="postgres",
    user="postgres",
    password="z@jreQbQRw35PW!"   # Supabase database password
)
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS route_fare_reliability (
    route VARCHAR(20) PRIMARY KEY,
    trip_count INT,
    avg_fare FLOAT,
    disruption_count INT,
    reliability_score FLOAT
)
''')

cursor.execute('DELETE FROM route_fare_reliability')
for _, row in merged.iterrows():
    cursor.execute(
        'INSERT INTO route_fare_reliability (route, trip_count, avg_fare, disruption_count, reliability_score) VALUES (%s, %s, %s, %s, %s)',
        (row['route'], int(row['trip_count']), float(row['avg_fare']), int(row['disruption_count']), float(row['reliability_score']))
    )

conn.commit()
cursor.execute('SELECT COUNT(*) FROM route_fare_reliability')
print('Rows inserted:', cursor.fetchone()[0])

### 14. Sample parameterised query (for report screenshot)

In [ ]:
# Example parameterised SELECT -- demonstrates safe query practice (no SQL injection risk)
threshold = 0.01
cursor.execute('SELECT route, avg_fare, reliability_score FROM route_fare_reliability WHERE reliability_score > %s ORDER BY reliability_score DESC', (threshold,))
results = cursor.fetchall()
for r in results:
    print(r)

cursor.close()
conn.close()